# Layout Detection Benchmark
**3 Models × 3 Datasets — Clean Evaluation Pipeline**

This notebook benchmarks three state-of-the-art layout detection models against three open-source datasets. All dataset loading, inference wrapping, evaluation metrics, and visualization utilities are imported from `backend.benchmark` to keep this notebook clean and focused.

### Models Under Evaluation
1. **DocLayoutYOLO**: Fast, CPU-capable layout detector from the `docling` package.
2. **NVIDIA Nemotron-Parse-v1.1**: ViT-H encoder + mBart decoder model for document layout parsing (requires CUDA GPU).
3. **LandingAI ADE DPT-2**: Document Parsing Transformer model accessed via LandingAI API (requires API Key).

### Datasets Under Evaluation
- **DocLayNet**: Sampled 30 pages from the test split (11 classes).
- **PubLayNet**: Sampled 30 pages from the val split (5 classes).
- **DocBank**: Sampled 30 pages from the test split (9 classes).

> **Runtime Notice**: Enable GPU (T4 or higher) in notebook settings before running to execute Nemotron-Parse-v1.1.

In [1]:
# ═══════════════════════════════════════════════════════
# 1. Configuration & Environment Setup
# ═══════════════════════════════════════════════════════
import os
import sys
import random
import numpy as np
import torch
import warnings
warnings.filterwarnings('ignore')

# Add repository root path to import backend modules
sys.path.append(os.path.abspath('.'))
from backend import benchmark

SEED = 42
N_SAMPLES = 30
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_DIR = './benchmark_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

LANDING_AI_KEY = "ZHdjNDh0eGkwcnk0bGZkb3Q0czZzOlNkNHdMQ1JsQWtBYVR4U3RiMUhSbDJmSjA5N2VPMXpk"
if LANDING_AI_KEY:
    print("✅ LandingAI key loaded directly.")
else:
    print("⚠️ No LandingAI key found. ADE DPT-2 will be skipped.")

print(f"Running on device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

✅ LandingAI key loaded directly.
Running on device: cuda
GPU: NVIDIA GeForce GTX 1650
VRAM: 4.3 GB


# ═══════════════════════════════════════════════════════
# 2. Dataset Loading & Sampling
# ═══════════════════════════════════════════════════════
We load the three open-source datasets, sample `N_SAMPLES = 30` pages from each using a fixed seed, and inspect the class distributions.

In [ ]:
samples = {}
samples['DocLayNet'] = benchmark.load_doclaynet_samples(n_samples=N_SAMPLES, seed=SEED)
samples['PubLayNet'] = benchmark.load_publaynet_samples(n_samples=N_SAMPLES, seed=SEED)
samples['DocBank'] = benchmark.load_docbank_samples(n_samples=N_SAMPLES, seed=SEED)

print("\n" + "="*60)
for ds_name, ds_samples in samples.items():
    class_dist = {}
    for s in ds_samples:
        for b in s['gt']:
            class_dist[b['label']] = class_dist.get(b['label'], 0) + 1
    print(f"{ds_name}: {len(ds_samples)} pages, classes: {dict(sorted(class_dist.items()))}")
print("="*60)

Loading DocLayNet (test split, streaming)...


DocLayNet:   0%|          | 0/30 [00:00<?, ?it/s]

# ═══════════════════════════════════════════════════════
# 3. Model Inference
# ═══════════════════════════════════════════════════════
We run inference for each of the three models on our sampled datasets. Detections are stored mapped to the canonical label schema.

### Model A — DocLayoutYOLO (Docling)

In [ ]:
print("Initializing DocLayoutYOLO (Docling)...")
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import InputFormat

try:
    converter = DocumentConverter(allowed_formats=[InputFormat.IMAGE])
except Exception:
    converter = DocumentConverter()

dl_preds = {'DocLayNet': {}, 'PubLayNet': {}, 'DocBank': {}}
for ds_name in ['DocLayNet', 'PubLayNet', 'DocBank']:
    print(f"\n▶ DocLayoutYOLO on {ds_name}...")
    for i, sample in enumerate(samples[ds_name]):
        dl_preds[ds_name][i] = benchmark.run_doclay_on_image(sample['image'], converter)
    total = sum(len(dl_preds[ds_name][i]) for i in range(N_SAMPLES))
    print(f"  ✅ {ds_name}: {total} total detections ({total/N_SAMPLES:.1f}/page)")

### Model B — NVIDIA Nemotron-Parse-v1.1

In [ ]:
nm_preds = {'DocLayNet': {}, 'PubLayNet': {}, 'DocBank': {}}

if DEVICE == 'cuda':
    print("Loading Nemotron-Parse-v1.1 (this takes ~35s)...")
    from transformers import AutoModel, AutoProcessor, AutoTokenizer, GenerationConfig
    
    MODEL_PATH = "nvidia/NVIDIA-Nemotron-Parse-v1.1"
    nemo_model = AutoModel.from_pretrained(
        MODEL_PATH, trust_remote_code=True,
        torch_dtype=torch.float16, low_cpu_mem_usage=True
    ).to(DEVICE).eval()
    
    nemo_processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True)
    nemo_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    nemo_gen_config = GenerationConfig.from_pretrained(MODEL_PATH)
    nemo_gen_config.max_new_tokens = 512
    
    for ds_name in ['DocLayNet', 'PubLayNet', 'DocBank']:
        print(f"\n▶ Nemotron on {ds_name}...")
        for i, sample in enumerate(samples[ds_name]):
            nm_preds[ds_name][i] = benchmark.run_nemotron_on_image(
                sample['image'], nemo_model, nemo_processor, nemo_tokenizer, nemo_gen_config, device=DEVICE
            )
        total = sum(len(nm_preds[ds_name][i]) for i in range(N_SAMPLES))
        print(f"  ✅ {ds_name}: {total} total detections ({total/N_SAMPLES:.1f}/page)")
        
    # Free VRAM to allow plotting and other memory-heavy ops
    del nemo_model, nemo_processor, nemo_tokenizer
    torch.cuda.empty_cache()
    print("\n✅ Nemotron inference complete. Model unloaded from VRAM.")
else:
    print("⚠️ CPU environment detected. Skipping Nemotron (requires CUDA GPU).")
    for ds_name in ['DocLayNet', 'PubLayNet', 'DocBank']:
        for i in range(N_SAMPLES):
            nm_preds[ds_name][i] = []

### Model C — LandingAI ADE DPT-2

In [ ]:
ade_preds = {'DocLayNet': {}, 'PubLayNet': {}, 'DocBank': {}}

if LANDING_AI_KEY:
    print("Initializing LandingAI ADE DPT-2...")
    from landingai_ade import LandingAIADE
    ade_client = LandingAIADE(apikey=LANDING_AI_KEY)
    
    for ds_name in ['DocLayNet', 'PubLayNet', 'DocBank']:
        print(f"\n▶ ADE DPT-2 on {ds_name}...")
        for i, sample in enumerate(samples[ds_name]):
            ade_preds[ds_name][i] = benchmark.run_ade_on_image(sample['image'], ade_client)
        total = sum(len(ade_preds[ds_name][i]) for i in range(N_SAMPLES))
        print(f"  ✅ {ds_name}: {total} total detections ({total/N_SAMPLES:.1f}/page)")
else:
    print("⚠️ No LandingAI API key found. Skipping ADE DPT-2.")
    for ds_name in ['DocLayNet', 'PubLayNet', 'DocBank']:
        for i in range(N_SAMPLES):
            ade_preds[ds_name][i] = []

# ═══════════════════════════════════════════════════════
# 4. Evaluation & Metrics Calculation
# ═══════════════════════════════════════════════════════
We evaluate the predictions against ground-truth annotations using localized canonical mappings and the COCO evaluation methodology (mAP@50 and mAP@50:95).

In [ ]:
IOU_THRESHOLDS = [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]
results = {}

for ds_name in ['DocLayNet', 'PubLayNet', 'DocBank']:
    results[ds_name] = {}
    gt_list = [s['gt'] for s in samples[ds_name]]
    valid_cls = benchmark.VALID_CLASSES_PER_DATASET[ds_name]
    
    print(f"\n{'='*60}")
    print(f"Evaluating: {ds_name} ({len(valid_cls)} classes)")
    print(f"{'='*60}")
    
    for model_name, pred_dict in [
        ('DocLayoutYOLO', dl_preds[ds_name]),
        ('Nemotron',      nm_preds[ds_name]),
        ('ADE-DPT2',     ade_preds[ds_name]),
    ]:
        pred_list = [pred_dict[i] for i in range(N_SAMPLES)]
        metrics = benchmark.compute_map(gt_list, pred_list,
                                      iou_thresholds=IOU_THRESHOLDS,
                                      valid_classes=valid_cls)
        results[ds_name][model_name] = metrics
        
        is_skipped = (model_name == 'ADE-DPT2' and not LANDING_AI_KEY) or \
                     (model_name == 'Nemotron' and DEVICE != 'cuda')
        status = '(N/A - skipped)' if is_skipped else ''
        
        print(f"  {model_name:16s}: "
              f"mAP@50={metrics['mAP50']:.3f}  "
              f"mAP@50:95={metrics['mAP5095']:.3f}  "
              f"P={metrics['precision']:.3f}  "
              f"R={metrics['recall']:.3f}  "
              f"F1={metrics['F1']:.3f}  "
              f"mIoU={metrics['mean_iou']:.3f}  "
              f"{status}")

# ═══════════════════════════════════════════════════════
# 5. Benchmark Visualizations
# ═══════════════════════════════════════════════════════
We render summary tables and heatmaps to compile quantitative model comparisons.

### styled 3×3 Summary Table

In [ ]:
df_summary = benchmark.plot_summary_table(results, N_SAMPLES, OUTPUT_DIR)

### Per-Class Precision Heatmap (DocLayNet)

In [ ]:
benchmark.plot_perclass_heatmap(results, 'DocLayNet', OUTPUT_DIR)

### Visual Comparison Samples

In [ ]:
for ds_name in ['DocLayNet', 'PubLayNet', 'DocBank']:
    idx = random.randint(0, N_SAMPLES - 1)
    preds_by_model = {
        'DocLayoutYOLO': dl_preds[ds_name],
        'Nemotron': nm_preds[ds_name],
        'ADE-DPT2': ade_preds[ds_name]
    }
    benchmark.plot_visual_comparison(samples[ds_name], preds_by_model, ds_name, idx, OUTPUT_DIR)

# ═══════════════════════════════════════════════════════
# 6. Export Results
# ═══════════════════════════════════════════════════════
We save the comprehensive raw evaluation metrics to a JSON file.

In [ ]:
import json

def make_serializable(obj):
    if isinstance(obj, dict):
        return {str(k): make_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [make_serializable(v) for v in obj]
    elif isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, set):
        return sorted(list(obj))
    return obj

json_path = os.path.join(OUTPUT_DIR, 'full_results.json')
with open(json_path, 'w') as f:
    json.dump(make_serializable(results), f, indent=2, default=str)

print(f"📁 Results saved to {OUTPUT_DIR}/")
print(f"   ├── benchmark_summary.csv")
print(f"   ├── full_results.json")
print(f"   ├── doclaynet_perclass_ap.png")
print(f"   ├── visual_sample_DocLayNet.png")
print(f"   ├── visual_sample_PubLayNet.png")
print(f"   └── visual_sample_DocBank.png")
print("\n🎉 Benchmark complete!")